In [0]:
import os
from pyspark.sql.functions import from_json, col, explode, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType

In [0]:
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("CONTAINER_NAME")
container_raw = os.getenv("CONTAINER_RAW")

In [0]:
spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        storage_account_key
    )  

In [0]:
path=f"abfss://{container_raw}@{storage_account_name}.dfs.core.windows.net/"

In [0]:
display(dbutils.fs.ls(path))

In [0]:
df = spark.read.json(path)

In [0]:
df.printSchema()

In [0]:
df.display()

In [0]:
amiibo_schema = ArrayType(
    StructType([
        StructField("amiiboSeries", StringType()),
        StructField("character", StringType()),
        StructField("gameSeries", StringType()),
        StructField("head", StringType()),
        StructField("image", StringType()),
        StructField("name", StringType()),
        StructField("release", StructType([
            StructField("au", StringType()),
            StructField("eu", StringType()),
            StructField("jp", StringType()),
            StructField("na", StringType())
        ])),
        StructField("tail", StringType()),
        StructField("type", StringType())
    ])
)


In [0]:
schema = StructType([
    StructField("amiibo", amiibo_schema)
])

In [0]:
df_parsed = df.withColumn("json_data", from_json(col("raw_content"), schema))

In [0]:
df_exploded = df_parsed.select(explode(col("json_data.amiibo")).alias("amiibo"))

In [0]:
df_exploded.display()

In [0]:
df_amiibos = df_exploded.select(
    col("amiibo.amiiboSeries").alias("amiibo_series"),
    col("amiibo.character"),
    col("amiibo.gameSeries").alias("game_series"),
    col("amiibo.head"),
    col("amiibo.image"),
    col("amiibo.name"),
    col("amiibo.release.au").alias("release_au"),
    col("amiibo.release.eu").alias("release_eu"),
    col("amiibo.release.jp").alias("release_jp"),
    col("amiibo.release.na").alias("release_na"),
    col("amiibo.tail"),
    col("amiibo.type")
)

In [0]:
df_amiibos.display()

In [0]:
df_amiibos.createOrReplaceTempView("amiibos")

In [0]:
path_save=f"abfss://amiibodata@adlsstoragejulie.dfs.core.windows.net/__unitystorage/bronze/amiibo_all_data"

In [0]:
df_amiibos = df_amiibos.withColumn("ingest_time", current_timestamp())

In [0]:
df_amiibos.display()

In [0]:
df_amiibos.write.mode("overwrite").saveAsTable("amiibo_data.bronze.amiibo_all")